In [1]:
# Ray Data ActorPool Underutilization Reproducer
# ================================================
# Reproduces: https://github.com/ray-project/ray/issues/XXXXX
#
# Shows that ActorPoolStrategy(initial_size=N, min_size=1) underutilizes
# the actor pool and delivers lower throughput than
# ActorPoolStrategy(min_size=N, max_size=N), even though both configurations
# have the same maximum number of actors.
#
# Tested on: Ray 2.55.1

import ray
import ray.data
import time, os, pprint, shutil, threading
from collections import Counter
from dataclasses import dataclass

print(f"Ray version: {ray.__version__}")


Ray version: 2.55.1


In [2]:
# ── Pipeline Configuration ────────────────────────────────────────────
VIDEO_DIR        = "/raid/curator-team/datasets/openvid-1m/video"
OUTPUT_DIR       = "/raid/weijiac/tmp/raydata_transcode_test"
VIDEO_LIMIT      = 1000       # number of videos to process
TRANSCODE_CPUS   = 6.0        # CPUs reserved per TranscodeActor
MAX_TRANSCODE_OVERRIDE = 9    # set to reproduce the observed underutilization run


In [3]:
import subprocess, tempfile, json as _json, uuid, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

CLIP_LEN_S = 10.0
MIN_CLIP_LEN_S = 2.0
ENCODE_BATCH_SIZE = 16
ENCODER = "libopenh264"
ENCODER_THREADS = 1
# TRANSCODE_CPUS set in cell 1
WRITE_CPUS = 0.25
CLIPS_PER_CHUNK = 32
CHUNK_SIZE_S = CLIPS_PER_CHUNK * 8 * CLIP_LEN_S

VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv", ".webm")


def ls_videos(batch):
    # Simulate slow upstream I/O (e.g. S3/NFS file listing, large dataset scan).
    import time as _time
    _time.sleep(15)
    paths = []
    for ext in VIDEO_EXTENSIONS:
        paths.extend(Path(VIDEO_DIR).rglob(f"*{ext}"))
    paths = sorted(str(p) for p in paths)[:VIDEO_LIMIT]
    return {"path": paths}

def read_and_probe(batch):
    out = {k: [] for k in ["path", "bytes", "duration", "fps", "num_frames",
                            "width", "height", "video_codec", "pixel_format",
                            "audio_codec", "bit_rate_k"]}
    for path in batch["path"]:
        raw = Path(path).read_bytes()
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
            tmp.write(raw)
            tmp_path = tmp.name
        try:
            result = subprocess.run(
                ["ffprobe", "-v", "error", "-show_format", "-show_streams",
                 "-of", "json", tmp_path],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
            )
        finally:
            Path(tmp_path).unlink(missing_ok=True)
        info = _json.loads(result.stdout)
        vs = next(s for s in info["streams"] if s["codec_type"] == "video")
        audio_codec = next((s["codec_name"] for s in info["streams"]
                            if s["codec_type"] == "audio"), None)
        num, den = map(int, vs["avg_frame_rate"].split("/"))
        fps = num / den
        duration = float(vs["duration"]) if "duration" in vs else float(info["format"]["duration"])
        bit_rate_k = int(int(vs["bit_rate"]) / 1024) if "bit_rate" in vs else 2000
        out["path"].append(path)
        out["bytes"].append(raw)
        out["duration"].append(duration)
        out["fps"].append(fps)
        out["num_frames"].append(int(duration * fps))
        out["width"].append(int(vs["width"]))
        out["height"].append(int(vs["height"]))
        out["video_codec"].append(vs["codec_name"])
        out["pixel_format"].append(vs["pix_fmt"])
        out["audio_codec"].append(audio_codec)
        out["bit_rate_k"].append(bit_rate_k)
    return out

def split_fixed_stride(batch):
    out = {k: [] for k in list(batch.keys()) + ["clips"]}
    for i in range(len(batch["path"])):
        duration = batch["duration"][i]
        clips, t = [], 0.0
        while t < duration:
            end = min(t + CLIP_LEN_S, duration)
            if end - t >= MIN_CLIP_LEN_S:
                clips.append((t, end))
            t += CLIP_LEN_S
        for k in batch.keys():
            out[k].append(batch[k][i])
        out["clips"].append(clips)
    return out

def _split_clips_into_chunks(clips_with_ids):
    chunks, cur_chunk, cur_dur = [], [], 0.0
    for item in clips_with_ids:
        (s, e), cid = item
        cur_chunk.append(item)
        cur_dur += e - s
        if cur_dur >= CHUNK_SIZE_S:
            chunks.append(cur_chunk)
            cur_chunk, cur_dur = [], 0.0
    if cur_chunk:
        chunks.append(cur_chunk)
    return chunks

def transcode(batch):
    out = {"path": [], "clip_chunk_index": [], "num_total_clips": [],
           "num_clip_chunks": [], "clips": [], "clip_bytes_map": [],
           "width": [], "height": [], "fps": [], "num_frames": [],
           "video_codec": [], "pixel_format": [], "audio_codec": [], "bit_rate_k": []}
    for i, path in enumerate(batch["path"]):
        # Ray Data passes batch values as numpy arrays — convert to Python types
        clips = [(float(s), float(e)) for s, e in batch["clips"][i]]
        if len(clips) == 0:
            continue
        pix_fmt = str(batch["pixel_format"][i]) if batch["pixel_format"][i] is not None else ""
        force_pix_fmt = "10le" in pix_fmt or "10be" in pix_fmt
        with tempfile.TemporaryDirectory() as tmp_dir:
            tmp = Path(tmp_dir)
            (tmp / "input.mp4").write_bytes(bytes(batch["bytes"][i]))
            fps = float(batch["fps"][i])
            clip_ids = [
                str(uuid.uuid5(uuid.NAMESPACE_URL, f"{path}_{int(s*fps)}_{int(e*fps)}"))
                for s, e in clips
            ]
            for b in range(0, len(clips), ENCODE_BATCH_SIZE):
                b_clips = clips[b:b+ENCODE_BATCH_SIZE]
                b_ids = clip_ids[b:b+ENCODE_BATCH_SIZE]
                cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error"]
                for j, (start, end) in enumerate(b_clips):
                    cmd += ["-threads", str(ENCODER_THREADS),
                            "-ss", str(start), "-to", str(end), "-i", "input.mp4",
                            "-map", f"{j}:v:0", "-c:v", ENCODER]
                    if force_pix_fmt:
                        cmd += ["-pix_fmt", "yuv420p"]
                    cmd += ["-threads", str(ENCODER_THREADS),
                            "-map", f"{j}:a:0?", "-c:a", "copy", f"{b_ids[j]}.mp4"]
                subprocess.check_output(cmd, cwd=tmp, stderr=subprocess.STDOUT)
            clip_bytes = {cid: (tmp / f"{cid}.mp4").read_bytes()
                          for cid in clip_ids if (tmp / f"{cid}.mp4").exists()}
        clip_chunks = _split_clips_into_chunks(list(zip(clips, clip_ids)))  # [(s,e), cid] pairs
        for chunk_idx, chunk in enumerate(clip_chunks):
            chunk_clips = [(s, e) for (s, e), _ in chunk]
            chunk_ids = [cid for _, cid in chunk]
            out["path"].append(path)
            out["clip_chunk_index"].append(chunk_idx)
            out["num_total_clips"].append(len(clips))
            out["num_clip_chunks"].append(len(clip_chunks))
            out["clips"].append(list(zip(chunk_ids, chunk_clips)))
            out["clip_bytes_map"].append({cid: clip_bytes[cid] for cid in chunk_ids if cid in clip_bytes})
            for k in ["width", "height", "fps", "num_frames", "video_codec",
                      "pixel_format", "audio_codec", "bit_rate_k"]:
                out[k].append(batch[k][i])
    return out

def _probe_clip(clip_bytes):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        tmp.write(clip_bytes)
        tmp_path = tmp.name
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_streams", "-of", "json", tmp_path],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True,
        )
    finally:
        Path(tmp_path).unlink(missing_ok=True)
    vs = next(s for s in _json.loads(result.stdout)["streams"] if s["codec_type"] == "video")
    num, den = map(int, vs["avg_frame_rate"].split("/"))
    return {"width": int(vs["width"]), "height": int(vs["height"]),
            "framerate": num/den, "num_frames": int(vs.get("nb_frames", 0)),
            "video_codec": vs["codec_name"], "num_bytes": len(clip_bytes)}

def write_clips(batch):
    clips_dir = Path(OUTPUT_DIR) / "clips"
    metas_dir = Path(OUTPUT_DIR) / "metas" / "v0"
    videos_dir = Path(OUTPUT_DIR) / "processed_videos"
    chunks_dir = Path(OUTPUT_DIR) / "processed_clip_chunks"
    for d in [clips_dir, metas_dir, videos_dir, chunks_dir]:
        d.mkdir(parents=True, exist_ok=True)

    def write_one_chunk(i):
        path = batch["path"][i]
        clip_bytes_map = batch["clip_bytes_map"][i]
        chunk_clips = [(str(cid), (float(s), float(e))) for cid, (s, e) in batch["clips"][i]]
        chunk_idx = int(batch["clip_chunk_index"][i])
        vmeta = {
            "width": int(batch["width"][i]),
            "height": int(batch["height"][i]),
            "fps": float(batch["fps"][i]),
            "num_frames": int(batch["num_frames"][i]),
            "video_codec": str(batch["video_codec"][i]),
            "pixel_format": str(batch["pixel_format"][i]),
            "audio_codec": str(batch["audio_codec"][i]) if batch["audio_codec"][i] is not None else None,
            "bit_rate_k": int(batch["bit_rate_k"][i]),
        }
        num_transcoded, total_dur, max_dur = 0, 0.0, 0.0
        with ThreadPoolExecutor(max_workers=6) as ex:
            futures = []
            for cid, (start, end) in chunk_clips:
                cb = clip_bytes_map.get(cid)
                if not cb:
                    continue
                futures.append(ex.submit((clips_dir / f"{cid}.mp4").write_bytes, cb))
                span_dur = end - start
                clip_meta = {
                    "span_uuid": cid, "source_video": path,
                    "duration_span": [start, end],
                    "width_source": vmeta["width"], "height_source": vmeta["height"],
                    "framerate_source": vmeta["fps"],
                    "clip_location": str(clips_dir / f"{cid}.mp4"),
                    "windows": [], "valid": True,
                }
                try:
                    clip_meta.update(_probe_clip(cb))
                except Exception:
                    pass
                futures.append(ex.submit(
                    (metas_dir / f"{cid}.json").write_text, _json.dumps(clip_meta)))
                num_transcoded += 1
                total_dur += span_dur
                max_dur = max(max_dur, span_dur)
            for f in futures:
                f.result()
        if chunk_idx == 0:
            video_meta = {
                "video": path, "height": vmeta["height"], "width": vmeta["width"],
                "framerate": vmeta["fps"], "num_frames": vmeta["num_frames"],
                "video_codec": vmeta["video_codec"], "pixel_format": vmeta["pixel_format"],
                "audio_format": vmeta["audio_codec"],
                "num_total_clips": int(batch["num_total_clips"][i]),
                "num_clip_chunks": int(batch["num_clip_chunks"][i]),
            }
            rel = path.removeprefix(VIDEO_DIR).lstrip("/") + ".json"
            vpath = videos_dir / rel
            vpath.parent.mkdir(parents=True, exist_ok=True)
            vpath.write_text(_json.dumps(video_meta))
        chunk_stats = {
            "video": path, "clip_chunk_index": chunk_idx,
            "num_clips_transcoded": num_transcoded, "num_clips_passed": num_transcoded,
            "total_clip_duration": total_dur, "max_clip_duration": max_dur,
            "clips": [cid for cid, _ in chunk_clips],
        }
        rel = path.removeprefix(VIDEO_DIR).lstrip("/") + f"_{chunk_idx}.json"
        cpath = chunks_dir / rel
        cpath.parent.mkdir(parents=True, exist_ok=True)
        cpath.write_text(_json.dumps(chunk_stats))
        return {"path": path, "clip_ids": [cid for cid, _ in chunk_clips]}

    results = [write_one_chunk(i) for i in range(len(batch["path"]))]
    return {"path": [r["path"] for r in results],
            "clip_ids": [r["clip_ids"] for r in results]}

In [4]:
# ── Pipeline Runs ────────────────────────────────────────────────────────────
import threading
from ray.data import ActorPoolStrategy, TaskPoolStrategy

ray.shutdown()
ray.init(num_cpus=64)  # 64-CPU machine

total_cpus = int(ray.cluster_resources().get("CPU", 0))
max_transcode = MAX_TRANSCODE_OVERRIDE
print(f"Total CPUs: {total_cpus}, max transcode workers: {max_transcode}")
print(f"CPU pressure: {max_transcode * TRANSCODE_CPUS:.0f}/{total_cpus} = "
      f"{max_transcode * TRANSCODE_CPUS / total_cpus * 100:.0f}%")

class TranscodeActor:
    def __call__(self, batch):
        return transcode(batch)

def _monitor_actors(snapshots, stop_event, interval=1.0):
    # Track active TranscodeActor count via CPU utilization.
    total_cpu = ray.cluster_resources().get("CPU", 64)
    t0 = time.perf_counter()
    while not stop_event.is_set():
        try:
            avail = ray.available_resources().get("CPU", total_cpu)
            busy = max(0.0, total_cpu - avail)
            actor_est = int(round(busy / TRANSCODE_CPUS))
            ts = time.perf_counter() - t0
            snapshots.append((ts, busy, actor_est))
        except Exception:
            pass
        stop_event.wait(interval)

def run_pipeline(transcode_compute, label):
    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

    snapshots = []
    stop_event = threading.Event()
    monitor_thread = threading.Thread(
        target=_monitor_actors, args=(snapshots, stop_event), daemon=True
    )

    t0 = time.perf_counter()
    ds = (
        ray.data.from_items([{"seed": None}])
        .map_batches(ls_videos, batch_size=1)  # single task
        .repartition(VIDEO_LIMIT)
        .map_batches(read_and_probe, batch_size=1, num_cpus=1)
        .map_batches(split_fixed_stride, batch_size=1, num_cpus=1)
        .map_batches(TranscodeActor, batch_size=1, num_cpus=TRANSCODE_CPUS,
                     compute=transcode_compute)
        .repartition(VIDEO_LIMIT * 20)
        .map_batches(write_clips, batch_size=1, num_cpus=WRITE_CPUS)
    )
    monitor_thread.start()
    results = ds.take_all()
    stop_event.set()
    monitor_thread.join(timeout=3)
    elapsed = time.perf_counter() - t0

    n_chunks = len(results)
    n_videos = len(set(r["path"] for r in results))
    print(f"[{label}] {n_videos} videos, {n_chunks} clip-chunks, "
          f"{elapsed:.1f}s — {n_videos/elapsed:.2f} vid/s")
    return elapsed, snapshots

elapsed1, snaps1 = run_pipeline(
    ActorPoolStrategy(min_size=1, max_size=max_transcode, initial_size=max_transcode),
    label="ActorPool(min=1, max=N, initial=N)"
)

elapsed2, snaps2 = run_pipeline(
    ActorPoolStrategy(min_size=max_transcode, max_size=max_transcode),
    label="ActorPool(min=N, max=N)"
)

2026-07-06 19:16:41,821	INFO worker.py:2003 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8268 
/opt/venv/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Total CPUs: 64, max transcode workers: 9
CPU pressure: 54/64 = 84%


2026-07-06 19:16:44,644	INFO logging.py:416 -- Registered dataset logger for dataset dataset_7_0
2026-07-06 19:16:44,664	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2026-07-06_19-16-28_966894_3085811/logs/ray-data
2026-07-06 19:16:44,665	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(ls_videos)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[MapBatches(read_and_probe)->MapBatches(split_fixed_stride)] -> ActorPoolMapOperator[MapBatches(TranscodeActor)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[MapBatches(write_clips)]
[2026-07-06 19:16:44,700 E 3085811 3085811] core_worker.cc:2194: Actor with class name: 'MapWorker(MapBatches(TranscodeActor))' and ID: '92944ed62242556e267e28ed01000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost,

[ActorPool(min=1, max=N, initial=N)] 1000 videos, 1000 clip-chunks, 360.6s — 2.77 vid/s


2026-07-06 19:22:45,147	INFO logging.py:416 -- Registered dataset logger for dataset dataset_15_0
2026-07-06 19:22:45,156	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_15_0. Full logs are in /tmp/ray/session_2026-07-06_19-16-28_966894_3085811/logs/ray-data
2026-07-06 19:22:45,158	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_15_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(ls_videos)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[MapBatches(read_and_probe)->MapBatches(split_fixed_stride)] -> ActorPoolMapOperator[MapBatches(TranscodeActor)] -> AllToAllOperator[Repartition] -> TaskPoolMapOperator[MapBatches(write_clips)]
2026-07-06 19:22:45,453	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_15_0 =======
2026-07-06 19:22:45,455	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-07-06 19:22:45,455	INFO logging_progress.py:227 -- Active & requested resources: 0/64 CPU, 0.0B/93.1GiB object store (

[ActorPool(min=N, max=N)] 1000 videos, 1000 clip-chunks, 332.0s — 3.01 vid/s


In [5]:
def print_actor_trace(label, snapshots, width=60):
    if not snapshots:
        print(f"[{label}] no snapshots captured")
        return
    times   = [s[0] for s in snapshots]
    busy    = [s[1] for s in snapshots]
    actors  = [s[2] for s in snapshots]
    peak    = max(actors) if actors else 0
    if peak == 0:
        print(f"[{label}] all snapshots show 0 actors (busy CPU always 0?)")
        # show busy cpu trace anyway
        peak_b = max(busy) if busy else 1
        scale = width / max(peak_b, 1)
        for ts, b, a in zip(times, busy, actors):
            bar = "█" * int(b * scale)
            print(f"  {ts:7.1f}s  cpu_busy={b:4.0f}  {bar}")
        return
    scale = width / peak
    for ts, b, a in zip(times, busy, actors):
        bar = "█" * int(a * scale)
        print(f"  {ts:7.1f}s  actors~{a:2d}  cpu_busy={b:4.0f}  {bar}")
    avg = sum(actors) / len(actors)
    zeros = sum(1 for a in actors if a == 0)
    print(f"  peak={peak}  avg={avg:.1f}  zero-actor-samples={zeros}/{len(actors)}")

print("\n=== Run 1: ActorPool(min=1, max=N, initial=N) ===")
print_actor_trace("min=1", snaps1)

print("\n=== Run 2: ActorPool(min=N, max=N) ===")
print_actor_trace("min=N", snaps2)

print()
print("=" * 72)
print("  SUMMARY")
print("=" * 72)
peak1 = max((s[2] for s in snaps1), default=0)
peak2 = max((s[2] for s in snaps2), default=0)
avg1  = sum(s[2] for s in snaps1) / len(snaps1) if snaps1 else 0
avg2  = sum(s[2] for s in snaps2) / len(snaps2) if snaps2 else 0
zero1 = sum(1 for s in snaps1 if s[2] == 0)
zero2 = sum(1 for s in snaps2 if s[2] == 0)
print(f"  {'Strategy':<35} {'time(s)':>8} {'peak':>6} {'avg':>6} {'zero%':>7}")
print(f"  {'-'*35} {'-'*8} {'-'*6} {'-'*6} {'-'*7}")
print(f"  {'ActorPool(min=1, max=N, initial=N)':<35} {elapsed1:>8.1f} {peak1:>6} {avg1:>6.1f} {100*zero1/len(snaps1) if snaps1 else 0:>6.1f}%")
print(f"  {'ActorPool(min=N, max=N)':<35} {elapsed2:>8.1f} {peak2:>6} {avg2:>6.1f} {100*zero2/len(snaps2) if snaps2 else 0:>6.1f}%")
speedup = elapsed1 / elapsed2 if elapsed2 else 0
print(f"\n  Speedup (min=N vs min=1): {speedup:.2f}x")



=== Run 1: ActorPool(min=1, max=N, initial=N) ===
      0.0s  actors~ 0  cpu_busy=   0  
      1.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      2.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      3.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      4.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      5.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      6.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      7.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      8.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
      9.0s  actors~ 9  cpu_busy=  55  ██████████████████████████████████████████████████████
     10.0s  actors~ 9  cpu_busy=  55  ███████████████████████████████████

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# OBSERVATION
# ══════════════════════════════════════════════════════════════════════════════
#
# We ran two configurations on a video transcoding pipeline:
#   Run 1: ActorPoolStrategy(initial_size=9, min_size=1, max_size=9)
#   Run 2: ActorPoolStrategy(min_size=9, max_size=9)
#
# Run 1 was slower despite having the same max actor count.
# The actor trace shows the pool drops from 9 and stabilizes around 7,
# while Run 2 stays at 9 throughout.
#
# We expect Run 1 to perform at least as well as Run 2.
#
# ── Autoscaler log (Run 1, from ray-data debug log) ─────────────────────────
#
#   Scaled up   actor pool by 9
#   Scaled down actor pool by 1  (running=8, then further to ~7)

print("=" * 72)
print("  OBSERVATION SUMMARY")
print("=" * 72)
summary = """
  Run 1  ActorPoolStrategy(initial_size=N, min_size=1, max_size=N):
    - Actor count dropped from 9 and stabilized around 7
    - Throughput: 2.77 vid/s  (360.6s total)

  Run 2  ActorPoolStrategy(min_size=N, max_size=N):
    - Actor count stable at 9 throughout
    - Throughput: 3.01 vid/s  (332.0s total)

  Expected: Run 1 >= Run 2 (same max actors).
  Observed: Run 1 ~9% slower.
"""
print(summary)


  OBSERVATION SUMMARY

  Run 1  ActorPoolStrategy(initial_size=N, min_size=1, max_size=N):
    - Actor count dropped from 9 and stabilized around 7
    - Throughput: 2.77 vid/s  (360.6s total)

  Run 2  ActorPoolStrategy(min_size=N, max_size=N):
    - Actor count stable at 9 throughout
    - Throughput: 3.01 vid/s  (332.0s total)

  Expected: Run 1 >= Run 2 (same max actors).
  Observed: Run 1 ~9% slower.

